In [1]:
import pandas as pd

In [4]:
df = pd.read_csv('20_datacenters_selectes_link_datacentermap.csv')

In [8]:
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

RAW_HTML_DIR = Path("raw_html/datacentermap_20")

In [11]:
def extract_page_props(path: Path) -> dict:
    """Lê um HTML do datacentermap.com e retorna o pageProps do __NEXT_DATA__."""
    html = path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")

    tag = soup.find("script", id="__NEXT_DATA__")
    if tag is None or tag.string is None:
        return {}

    payload = json.loads(tag.string)
    return payload.get("props", {}).get("pageProps", {})


def build_row(id_datacenter: str, page_props: dict) -> dict:
    dc = page_props.get("dc") or {}
    company = dc.get("companies") or {}
    services = (page_props.get("serviceplan") or {}).get("services") or {}
    colo = services.get("colo") or {}
    cloud = services.get("cloud") or {}

    meta_standards = dc.get("meta_standards") or {}
    meta_power = dc.get("meta_power") or {}
    meta_security = dc.get("meta_security") or {}
    meta_building = dc.get("meta_building") or {}
    meta_capacity = dc.get("meta_capacity") or {}
    meta_stats = dc.get("meta_stats") or {}
    meta_references = dc.get("meta_references") or {}

    if not dc:
        print(f"  ⚠️  {id_datacenter}: 'dc' vazio/nulo — verifique o HTML dessa página")

    link_datacenter = None
    if dc.get("countrylink") and dc.get("marketlink") and dc.get("link"):
        link_datacenter = f"https://www.datacentermap.com/{dc['countrylink']}/{dc['marketlink']}/{dc['link']}/"

    return {
        "id_datacenter": id_datacenter,
        "nome_datacenter": dc.get("name"),
        "operadora": company.get("name"),
        "endereco": dc.get("address"),
        "cep": dc.get("postal"),
        "cidade": dc.get("city"),
        "estado": dc.get("state"),
        "mercado": dc.get("market"),
        "pais": dc.get("country"),
        "latitude": dc.get("latitude"),
        "longitude": dc.get("longitude"),
        "descricao": dc.get("description"),
        "status": dc.get("status"),
        "stage": dc.get("stage"),
        "tipo_listagem": dc.get("listingtype"),
        "tipo_capacidade": dc.get("capacitytype"),
        "tags": ", ".join(dc.get("tags") or []),
        "link_datacenter": link_datacenter,

        # capacidade
        "mw_construido": meta_capacity.get("mw_builtout"),
        "mw_referenciado": meta_capacity.get("mw_referenced"),
        "whitespace_construido_sqm": meta_capacity.get("whitespace_builtout"),
        "whitespace_referenciado_sqm": meta_capacity.get("whitespace_referenced"),

        # prédio
        "ano_operacional": meta_building.get("year_operational"),
        "tipo_construcao": meta_building.get("construction"),
        "tipo_ocupacao": meta_building.get("tenancy_text"),
        "carga_piso_max_kg_sqm": meta_building.get("floor_load"),

        # energia / refrigeração
        "redundancia_refrigeracao": meta_power.get("cooling_redundancy"),

        # segurança
        "cctv": meta_security.get("cctv"),
        "controle_acesso_cartao": meta_security.get("keycard"),
        "biometria": meta_security.get("biometric"),

        # compliance / certificações
        "tier_projetado": meta_standards.get("tier_designed"),
        "tier_certificado": meta_standards.get("tier_certified"),
        "pci_dss": meta_standards.get("pci"),
        "iso9001": meta_standards.get("iso9001"),
        "iso14001": meta_standards.get("iso14001"),
        "iso22301": meta_standards.get("iso22301"),
        "iso27001": meta_standards.get("iso27001"),
        "iso45001": meta_standards.get("iso45001"),
        "iso50001": meta_standards.get("iso50001"),
        "soc1": meta_standards.get("soc1"),
        "soc2": meta_standards.get("soc2"),
        "soc3": meta_standards.get("soc3"),

        # serviços de colocation
        "colo_suites": colo.get("suites"),
        "colo_cages": colo.get("cages"),
        "colo_cabinets": colo.get("cabinets"),
        "colo_partial_cabinets": colo.get("partialcabinets"),
        "colo_shared_rackspace": colo.get("sharedrackspace"),
        "colo_footprints": colo.get("footprints"),
        "colo_remote_hands": colo.get("remotehands"),
        "colo_build_to_suit": colo.get("build"),

        # serviços de cloud
        "cloud_gpu": cloud.get("gpu"),
        "cloud_managed": cloud.get("managed"),
        "cloud_baremetal": cloud.get("baremetal"),
        "cloud_public_cloud": cloud.get("publiccloud"),

        # estatísticas
        "qtd_ixps": meta_stats.get("ixps"),
        "qtd_clouds": meta_stats.get("clouds"),
        "qtd_redes_presentes": meta_stats.get("networkpresence"),
        "qtd_provedores_rede": meta_stats.get("networkproviders"),
        "qtd_provedores_servico": meta_stats.get("serviceproviders"),

        # referências
        "codigo_site": meta_references.get("provider_id"),
        "peeringdb_id": meta_references.get("peeringdb_id"),
    }



In [ ]:
rows = {}

for path in sorted(RAW_HTML_DIR.glob("*.html")):
    if path.stem.endswith("_overview"):
        id_datacenter = path.stem.removesuffix("_overview")
        tipo_pagina = "overview"
    elif path.stem.endswith("_specs"):
        id_datacenter = path.stem.removesuffix("_specs")
        tipo_pagina = "specs"
    else:
        continue

    page_props = extract_page_props(path)
    if not page_props:
        print(f"Sem __NEXT_DATA__ em {path.name}, pulando.")
        continue

    row = build_row(id_datacenter, page_props)

    # os dados do dc são iguais em overview e specs; specs prevalece se já existir a chave
    if id_datacenter not in rows or tipo_pagina == "specs":
        rows[id_datacenter] = row

df_datacenters = pd.DataFrame(rows.values()).sort_values("id_datacenter").reset_index(drop=True)
print(f"Total: {len(df_datacenters)} data centers extraídos")
df_datacenters.head()



Sem __NEXT_DATA__ em dc_dced39dd76_specs.html, pulando.
Total: 21 data centers extraídos


In [ ]:
df_datacenters.to_csv("datacentermap_scraping_overview_specs.csv", index=False, encoding="utf-8-sig", sep=";")
#TODO: ajustar caminho para salvar